# Chapter 9 — Escalation and Human-in-the-Loop · Homework (self-test)

**Goal:** check that you can choose escalation triggers, enforce rules in code, run approvals and durable resumes safely, write a fact-checked handoff, and calibrate and audit confidence-based routing, without looking at the answers.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 90–120 minutes
- **Estimated API cost:** $0. Every graded part runs offline; agent runs use the SDK's `ScriptedModel`.
- **Prerequisites:** the repo venv from [00-prerequisites](../../00-prerequisites/README.md). No API key is needed.

### How to use this notebook

1. Answer the **quiz** (Part A) in the `answers` cell, then run the grader. Answers are stored as hashes, so you cannot peek.
2. Solve each **coding exercise** (Part B): replace the `TODO` stub, then run its check cell. A check never crashes the notebook; it prints ✓ or a hint.
3. Write your answer to the **architecture scenario** (Part C) and compare it with the rubric.
4. Run the **scorecard**. The pass mark is **72%** (quiz questions + exercises combined), the same bar as the Claude path.
5. Solutions are at the very bottom. Try first.

In [ ]:
import hashlib
import json
import math
import os
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from types import SimpleNamespace
from typing import Literal

from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, Field

from agents import Agent, Runner, RunState, StopAtTools, function_tool, set_tracing_disabled
from agents.testing import ScriptedModel, assistant_message, function_call

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
set_tracing_disabled(True)   # graded runs are offline
EX_RESULTS: dict[int, bool] = {}
QUIZ_SCORE: dict[str, int] = {"correct": 0, "total": 10}


def record(n: int, passed: bool, hint: str = "") -> None:
    EX_RESULTS[n] = passed
    print(f"✓ Exercise {n} passed" if passed else f"✗ Exercise {n} not passed yet. {hint}")


# Shared types used by the exercises
class HumanHandoff(BaseModel):
    customer_id: str
    order_id: str | None
    escalation_reason: Literal["customer_requested_human", "policy_gap", "policy_limit",
                               "cannot_progress", "identity_ambiguous"]
    issue_summary: str = Field(description="2-3 sentences a human can act on without the transcript")
    customer_request: str
    actions_taken: list[str]
    amount_usd: float | None
    recommended_action: str
    customer_sentiment_note: str | None


@dataclass
class DeskCtx:
    customer_id: str
    account_age_days: int
    auto_limit_usd: float = 100.0


print("Model (not used by graded parts):", MODEL)

## Part A — Quiz (10 questions)

**Q1.** A customer writes "change the delivery address for Maria Lopez". `find_customer` returns three customers with that name. What should the agent do?

- A) Pick the oldest of the three accounts, since long-time customers write in most often.
- B) Escalate to a human immediately with reason `identity_ambiguous`.
- C) Ask for another identifier (order ID or e-mail) and never pick one of the matches itself.
- D) Let the model choose the match whose city is most similar to the new address.

**Q2.** After calling `escalate_to_human`, a support agent keeps going: "I've passed this to a colleague, but meanwhile I issued a partial refund." The escalate tool is a strict `function_tool`. What is the most reliable fix?

- A) Add "After escalating, stop and do nothing else" to the instructions.
- B) `tool_use_behavior=StopAtTools(stop_at_tool_names=["escalate_to_human"])`, so the run ends with the tool's output as `final_output`.
- C) Set `max_turns=1` on every run.
- D) Set `tool_choice="required"` so the model must call a tool on every turn.

**Q3.** An agent sometimes loops on clarifying questions and lookups. With `max_turns=10`, customers then get a generic 500 error page because `MaxTurnsExceeded` is raised. What should the architect change?

- A) Raise `max_turns` to 50 so the agent has room to finish.
- B) Remove `max_turns` and rely on the model to stop.
- C) Catch the exception and re-run the same input from scratch.
- D) Pass `error_handlers={"max_turns": handler}` to `Runner.run`, where the handler opens a ticket from the run data and returns a controlled final output, and count clarifying questions in code.

**Q4.** A Triage agent hands off to a Refund specialist. The team attached an input guardrail that detects explicit requests for a human to the **Refund specialist**. In production it never fires. Why?

- A) Input guardrails run only for the first agent of a run; attach it to Triage (or pass it in `RunConfig(input_guardrails=...)`).
- B) Input guardrails need `run_in_parallel=False` to run at all.
- C) Guardrails only run with `Runner.run_streamed`.
- D) Handoffs strip all guardrails from the receiving agent's tools.

**Q5.** A supervisor rejects a $300 refund. When the run resumes, the model tells the customer "the tool failed, please try again later" and calls `issue_refund` again. The team wants the model to know a supervisor declined and to offer escalation instead. What fits best?

- A) Delete the pending item from the serialized state, then resume.
- B) Send a new user message "the refund was rejected" with `Runner.run(agent, "rejected")`.
- C) `state.reject(item, rejection_message="A supervisor declined this refund. Do not retry; offer to escalate.")`, then resume the same run.
- D) Approve the call and make the tool return an error string.

**Q6.** A bulk-update run calls `tag_order` (with `needs_approval=True`) 40 times. The reviewer approves the first call with `state.approve(item, always_approve=True)`. What is the scope of that decision?

- A) Every future run of any agent that uses `tag_order`.
- B) Every tool in this run.
- C) Only this one call ID.
- D) Later calls of `tag_order` for the rest of this run, including after the state is serialized and restored.

**Q7.** Paused runs are stored as `state.to_string()` in a review database. A security scan finds CRM API tokens in that database. The run context is a dataclass with `customer_id` and `crm_api_token`. What is the right fix?

- A) Pass `context_override` when loading the snapshot.
- B) Keep secrets out of the run context (load them inside the tool from a secret store), or serialize with a `context_serializer` that drops them; rebuild the context with `context_override` on resume.
- C) Base64-encode the snapshot before storing it.
- D) Serialize with `strict_context=True`.

**Q8.** A reviewer double-clicks "Approve". Two requests load the same stored snapshot, both resume, and the customer receives two goodwill vouchers. What prevents this?

- A) An atomic `pending → resuming` transition on the server-side review row before loading the snapshot, so each review is consumed exactly once.
- B) Passing `always_approve=False` explicitly.
- C) Checking the snapshot's `$schemaVersion` before resuming.
- D) `ToolExecutionConfig(pre_approval_tool_input_guardrails=True)`.

**Q9.** An invoice extractor reports a confidence per field. Product wants "auto-approve when confidence ≥ 0.9". On a labeled set, amounts scored 0.9–1.0 are 99% correct, but IBANs scored 0.9–1.0 are only 81% correct. What is the right design?

- A) One global threshold of 0.95 for all fields.
- B) Ask the model to double-check IBANs and keep 0.9.
- C) Drop confidence scores and send every document to review.
- D) Per-field thresholds picked on the labeled set (target precision, minimum support), with IBANs routed to review if no threshold meets the target; re-check on a held-out split.

**Q10.** A Responses API call with a remote `mcp` tool returns an `mcp_approval_request` item with `id` `mcpr_9` for `update_contact`. A human approves it. How does the application continue?

- A) Call the MCP server directly and send the result back as a `function_call_output` with `call_id` `mcpr_9`.
- B) A new `responses.create` with `previous_response_id`, the full `mcp` tool config again (including `authorization`), and input `[{"type": "mcp_approval_response", "approval_request_id": "mcpr_9", "approve": true}]`.
- C) Switch the tool to `"require_approval": "never"` and resend the original input.
- D) Resend the original request with `tool_choice="required"`.

In [ ]:
# Fill in "A", "B", "C" or "D" for each question.
answers = {
    "q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
    "q6": None, "q7": None, "q8": None, "q9": None, "q10": None,
}

In [ ]:
ANSWER_KEY = {
    "q1": "227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e",
    "q2": "61e5302385b2dd8204dbddfd86b357cacb0de07dbc800460aa335d9bc96b6ca1",
    "q3": "14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6",
    "q4": "0eca085218afe99ae6fb326d50c6e730cd71c102ccfe2f9a10d8e6054e2018e0",
    "q5": "e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f",
    "q6": "70425e0bfb4a573a030df7909d2bae441e3fdb1e7a565e097f3118231f4f27a9",
    "q7": "16dd052a869d89a87df9ae09526a5477a3e6870575379b0c9d2ed35de3ab21c7",
    "q8": "f98cc196593da1da1a7b4550be81b590cd44e7098497c2034a83c78f8a98f3bc",
    "q9": "e58517837d895fe6365a5801be9a66d9d1d39f1d863920a84cbed549cfec1954",
    "q10": "c69a9867de9de4cc2e45f4cd6a6128833af714d8993edbeac622795c9c5e20d3",
}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_KEY, key=lambda q: int(q[1:])):
        given = str(answers.get(qid) or "").strip().upper()
        ok = hashlib.sha256(f"{qid}:{given}".encode()).hexdigest() == ANSWER_KEY[qid]
        correct += ok
        print(f"{qid:>4}: {'✓' if ok else '✗'}  (your answer: {given or '-'})")
    QUIZ_SCORE["correct"] = correct
    print(f"\nQuiz: {correct}/{len(ANSWER_KEY)} = {correct / len(ANSWER_KEY):.0%}")
    return correct


quiz_correct = grade_quiz(answers)

## Part B — Coding exercises (offline)

### Exercise 1 — A rule-based escalation router

Write `route(case)` for a support agent. `case` is a dict with `text`, `topic`, `clarifications` (questions already asked), `matches` (customers found by the lookup) and `refund_usd`. Return exactly one route, checking in this order:

1. `"escalate:customer_requested_human"` when the text **asks** for a human: one of the verbs/phrases *want, need, talk to, speak to, speak with, get me, connect me, transfer me* followed (within the same sentence) by *human, person, manager* or *supervisor*. "My manager bought this" is **not** a request. Case-insensitive.
2. `"ask_identifier"` when `matches > 1`.
3. `"escalate:policy_gap"` when `topic` is not in `COVERED_TOPICS`.
4. `"escalate:cannot_progress"` when `clarifications >= 2`.
5. `"escalate:policy_limit"` when `refund_usd > 500`; `"approval"` when `refund_usd > 100`.
6. Otherwise `"resolve"`. Sentiment is **not** a trigger.

In [ ]:
COVERED_TOPICS = {"late_delivery", "damaged", "refund", "address_change", "duplicate_charge", "order_status"}


def route(case: dict) -> str:
    # TODO: implement the rules above, in order
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    def c(text, topic="refund", clarifications=0, matches=1, refund_usd=0.0):
        return {"text": text, "topic": topic, "clarifications": clarifications, "matches": matches, "refund_usd": refund_usd}

    cases = [
        (c("This is ridiculous, I'm SO angry! My parcel is late.", "late_delivery"), "resolve",
         "anger alone is not a trigger: acknowledge and resolve"),
        (c("You charged me twice, worst service ever!!!", "duplicate_charge", refund_usd=39), "resolve",
         "an angry but simple duplicate charge should be resolved"),
        (c("I want to talk to a human."), "escalate:customer_requested_human", "explicit request for a human"),
        (c("Can I SPEAK WITH a supervisor, please?"), "escalate:customer_requested_human", "matching must be case-insensitive"),
        (c("My manager bought this for me, can I return it?", refund_usd=39), "resolve",
         "'my manager bought this' is not a request for a human (no asking verb before it)"),
        (c("I want to talk to a manager", matches=3), "escalate:customer_requested_human",
         "an explicit request wins over every other rule"),
        (c("Change the address, I'm Maria Lopez", "address_change", matches=3), "ask_identifier",
         "several matches: ask for another identifier, never guess"),
        (c("Match the competitor's price", "price_match"), "escalate:policy_gap", "topic not covered by policy"),
        (c("Our lawyers will contact you", "legal_threat", refund_usd=900), "escalate:policy_gap",
         "policy gap is checked before refund limits"),
        (c("the thing... fix it", clarifications=2), "escalate:cannot_progress", "two clarifying questions already asked"),
        (c("Refund all $640, it's broken", "damaged", refund_usd=640), "escalate:policy_limit", "above the $500 hard limit"),
        (c("Refund the $250 headphones", refund_usd=250), "approval", "above $100 needs approval"),
        (c("Refund exactly $100", refund_usd=100), "resolve", "$100 is not above the auto limit"),
        (c("Refund exactly $500", refund_usd=500), "approval", "$500 is not above the hard limit, but above $100"),
    ]
    try:
        for case, want, why in cases:
            got = route(dict(case))
            if got != want:
                record(1, False, f"{case['text']!r}: expected {want!r}, got {got!r} ({why}).")
                return
        record(1, True)
    except NotImplementedError:
        record(1, False, "Implement route first.")
    except Exception as e:  # never crash the notebook
        record(1, False, f"{type(e).__name__}: {e}")


check_ex1()

### Exercise 2 — A fail-closed per-call approval policy

Write the async `needs_approval` callable for a refund tool: `refund_needs_approval(ctx, params, call_id) -> bool`. `ctx.context` is a `DeskCtx`. Return `True` (a human must approve) when **any** of these holds:

- `params["amount_usd"]` is missing, not a number (a `bool` is not a number), `NaN`, infinite, or negative: **fail closed**, never raise;
- the amount is above `ctx.context.auto_limit_usd`;
- the customer's account is younger than 30 days (`ctx.context.account_age_days < 30`).

Otherwise return `False`. The check also wires your policy into a real `function_tool(needs_approval=...)` run.

In [ ]:
async def refund_needs_approval(ctx, params: dict, call_id: str) -> bool:
    # TODO
    raise NotImplementedError

In [ ]:
async def check_ex2() -> None:
    try:
        old = SimpleNamespace(context=DeskCtx("C-1", account_age_days=400))
        new = SimpleNamespace(context=DeskCtx("C-2", account_age_days=5))
        strict = SimpleNamespace(context=DeskCtx("C-3", account_age_days=400, auto_limit_usd=20.0))
        table = [
            (old, {"order_id": "A", "amount_usd": 39.0}, False, "a small refund for an established account runs automatically"),
            (old, {"order_id": "A", "amount_usd": 100.0}, False, "exactly the limit is not above it"),
            (old, {"order_id": "A", "amount_usd": 100.01}, True, "above auto_limit_usd needs approval"),
            (strict, {"order_id": "A", "amount_usd": 39.0}, True, "use ctx.context.auto_limit_usd, not a hard-coded 100"),
            (new, {"order_id": "A", "amount_usd": 5.0}, True, "accounts younger than 30 days always need approval"),
            (old, {"order_id": "A"}, True, "a missing amount must fail closed (return True), not raise"),
            (old, {"order_id": "A", "amount_usd": "lots"}, True, "a non-numeric amount must fail closed"),
            (old, {"order_id": "A", "amount_usd": True}, True, "a bool is not an amount: fail closed"),
            (old, {"order_id": "A", "amount_usd": float("nan")}, True, "NaN must fail closed (comparisons with NaN are always False)"),
            (old, {"order_id": "A", "amount_usd": float("inf")}, True, "infinity must fail closed"),
            (old, {"order_id": "A", "amount_usd": -50.0}, True, "a negative refund must fail closed"),
        ]
        for ctx, params, want, why in table:
            try:
                got = await refund_needs_approval(ctx, dict(params), "call_1")
            except NotImplementedError:
                raise
            except Exception as e:
                record(2, False, f"{params} raised {type(e).__name__}; {why}.")
                return
            if got is not want:
                record(2, False, f"{params} with account_age_days={ctx.context.account_age_days}, "
                                 f"limit={ctx.context.auto_limit_usd}: expected {want}, got {got!r} ({why}).")
                return
        # Integration: the same policy inside a real Agents SDK run
        ran: list[float] = []

        @function_tool(needs_approval=refund_needs_approval)
        def issue_refund(order_id: str, amount_usd: float) -> str:
            """Refund an order."""
            ran.append(amount_usd)
            return "ok"

        for amount, should_pause in ((39.0, False), (300.0, True)):
            model = ScriptedModel([[function_call("issue_refund", {"order_id": "A", "amount_usd": amount}, call_id="r1")],
                                   [assistant_message("done")]])
            agent = Agent(name="Desk", model=model, tools=[issue_refund])
            result = await Runner.run(agent, "refund", context=DeskCtx("C-1", account_age_days=400))
            if bool(result.interruptions) is not should_pause:
                record(2, False, f"in a real run, ${amount} {'should' if should_pause else 'should not'} pause for approval.")
                return
        record(2, True)
    except NotImplementedError:
        record(2, False, "Implement refund_needs_approval first.")
    except Exception as e:
        record(2, False, f"{type(e).__name__}: {e}")


await check_ex2()

### Exercise 3 — A safe decision applier for a review queue

Reviewers decide on paused runs through your web app. Write `apply_review(store, review_id, reviewer, decisions, now)`. `store` maps review IDs to server-side records `{"status", "customer_id", "pending_call_ids", "expires_at"}`; `decisions` maps call IDs to `True`/`False` and comes from the browser. Check in this order:

1. Unknown `review_id` → raise `KeyError`.
2. `reviewer["role"] != "supervisor"`, or `reviewer["user_id"] == record["customer_id"]` (separation of duties) → raise `PermissionError`.
3. `record["status"] != "pending"` (already consumed: a replay or double click) → raise `RuntimeError`.
4. Any call ID in `decisions` that is not in `pending_call_ids` → raise `ValueError` and leave the status unchanged.
5. `now > record["expires_at"]` → set status `"expired"` and return `{call_id: "reject"}` for every pending call (fail closed).
6. Otherwise set status `"resuming"` and return `{call_id: "approve" | "reject"}` for **every** pending call: `"approve"` only when the decision is exactly `True`; a missing decision is `"reject"`.

In [ ]:
def apply_review(store: dict, review_id: str, reviewer: dict, decisions: dict, now: datetime) -> dict[str, str]:
    # TODO
    raise NotImplementedError

In [ ]:
async def check_ex3() -> None:
    t0 = datetime(2026, 10, 3, 9, 0, tzinfo=timezone.utc)
    sup = {"user_id": "sup_ana", "role": "supervisor"}

    def fresh():
        return {"rv1": {"status": "pending", "customer_id": "C-42", "pending_call_ids": ["c1", "c2"],
                        "expires_at": t0 + timedelta(hours=24)}}

    def expect_raise(exc_type, fn, hint):
        try:
            fn()
        except exc_type:
            return None
        except NotImplementedError:
            raise
        except Exception as e:
            return f"{hint} Expected {exc_type.__name__}, got {type(e).__name__}."
        return f"{hint} Expected {exc_type.__name__}, nothing was raised."

    try:
        store = fresh()
        problems = [
            expect_raise(KeyError, lambda: apply_review(store, "nope", sup, {}, t0), "Unknown review ID."),
            expect_raise(PermissionError, lambda: apply_review(fresh(), "rv1", {"user_id": "bo", "role": "agent"}, {"c1": True}, t0),
                         "Only supervisors may decide."),
            expect_raise(PermissionError, lambda: apply_review(fresh(), "rv1", {"user_id": "C-42", "role": "supervisor"}, {"c1": True}, t0),
                         "Separation of duties: the reviewer is the customer."),
        ]
        store = fresh()
        problems.append(expect_raise(ValueError, lambda: apply_review(store, "rv1", sup, {"c1": True, "c9": True}, t0),
                                     "Decisions for call IDs that are not pending must be refused."))
        if store["rv1"]["status"] != "pending" and not any(problems):
            problems.append("After refusing unknown call IDs, the review must stay 'pending'.")
        if any(problems):
            record(3, False, next(p for p in problems if p))
            return
        store = fresh()
        plan = apply_review(store, "rv1", sup, {"c1": True}, t0 + timedelta(hours=1))
        if plan != {"c1": "approve", "c2": "reject"}:
            record(3, False, f"Expected {{'c1': 'approve', 'c2': 'reject'}} (missing decision = reject), got {plan}.")
            return
        if store["rv1"]["status"] != "resuming":
            record(3, False, "A successful claim must move the status to 'resuming'.")
            return
        replay = expect_raise(RuntimeError, lambda: apply_review(store, "rv1", sup, {"c1": True}, t0 + timedelta(hours=1)),
                              "A second decision on the same review is a replay.")
        if replay:
            record(3, False, replay)
            return
        store = fresh()
        plan = apply_review(store, "rv1", sup, {"c1": "yes", "c2": 1}, t0)
        if plan != {"c1": "reject", "c2": "reject"}:
            record(3, False, f"Only a decision that is exactly True approves; got {plan}.")
            return
        store = fresh()
        plan = apply_review(store, "rv1", sup, {"c1": True, "c2": True}, t0 + timedelta(hours=30))
        if plan != {"c1": "reject", "c2": "reject"} or store["rv1"]["status"] != "expired":
            record(3, False, f"An expired review fails closed: all 'reject' and status 'expired'. Got {plan}, status {store['rv1']['status']!r}.")
            return
        # Integration: apply the plan to a real paused run, twice; the voucher must be issued once.
        vouchers: list[str] = []

        @function_tool(needs_approval=True)
        def issue_voucher(customer_id: str) -> str:
            """Issue a goodwill voucher."""
            vouchers.append(customer_id)
            return "ok"

        model = ScriptedModel([[function_call("issue_voucher", {"customer_id": "C-42"}, call_id="c1")],
                               [assistant_message("sent")], [assistant_message("sent again")]])
        agent = Agent(name="Goodwill", model=model, tools=[issue_voucher])
        blob = (await Runner.run(agent, "voucher please")).to_state().to_string()
        store = {"rv2": {"status": "pending", "customer_id": "C-42", "pending_call_ids": ["c1"], "expires_at": t0 + timedelta(hours=24)}}
        for _ in range(2):                                   # a double click
            try:
                plan = apply_review(store, "rv2", sup, {"c1": True}, t0)
            except RuntimeError:
                continue
            state = await RunState.from_string(agent, blob)
            for item in state.get_interruptions():
                state.approve(item) if plan.get(item.call_id) == "approve" else state.reject(item)
            await Runner.run(agent, state)
        if vouchers != ["C-42"]:
            record(3, False, f"A double click issued {len(vouchers)} vouchers; consume each review once.")
            return
        record(3, True)
    except NotImplementedError:
        record(3, False, "Implement apply_review first.")
    except Exception as e:
        record(3, False, f"{type(e).__name__}: {e}")


await check_ex3()

### Exercise 4 — Finalize the handoff from the run

The model drafted a `HumanHandoff` when it called `escalate_to_human`. Write `finalize_handoff(draft, customer_id, new_items, known_orders)` that returns a validated `HumanHandoff` where:

- `customer_id` is the authenticated `customer_id` argument, whatever the model wrote;
- `actions_taken` is rebuilt from `new_items` (a run's `result.new_items`), in order: for each `tool_call_item` that is **not** `escalate_to_human`, the string `f"Called {name}({arguments})"`; for each `tool_call_output_item`, `f"  -> {str(item.output)[:200]}"`, except outputs of the `escalate_to_human` call itself. Read the call name and arguments from `item.raw_item.name` / `item.raw_item.arguments`; an output item's `item.raw_item` is a dict with a `"call_id"`. Only function calls (`item.raw_item.type == "function_call"`) have a name and arguments. Hosted tools such as web search also produce `tool_call_item`s, so skip anything else (this run contains only function calls);
- `order_id` becomes `None` when it is not in `known_orders` (the authenticated customer's own order IDs, so another customer's order is dropped too);
- every other field stays as the model wrote it.

In [ ]:
def finalize_handoff(draft: dict, customer_id: str, new_items, known_orders: set[str]) -> HumanHandoff:
    # TODO
    raise NotImplementedError

In [ ]:
async def check_ex4() -> None:
    try:
        @function_tool
        def lookup_order(order_id: str) -> str:
            """Look up an order."""
            return json.dumps({"status": "delivered", "total_usd": 640.0})

        @function_tool
        def issue_refund(order_id: str, amount_usd: float) -> str:
            """Refund an order."""
            return json.dumps({"error": "needs_human", "message": "Above the $500 limit"})

        @function_tool
        def escalate_to_human(handoff: HumanHandoff) -> str:
            """Hand the case to a human."""
            return json.dumps({"status": "handoff_received"})

        draft = {"customer_id": "C-99", "order_id": "A-2O02", "escalation_reason": "policy_limit",
                 "issue_summary": "Damaged espresso machine, wants $640 back.", "customer_request": "Full refund",
                 "actions_taken": ["Issued a $640 refund"], "amount_usd": 640.0,
                 "recommended_action": "Approve refund", "customer_sentiment_note": "Frustrated"}
        model = ScriptedModel([
            [function_call("lookup_order", {"order_id": "A-2002"}, call_id="l1")],
            [function_call("issue_refund", {"order_id": "A-2002", "amount_usd": 640.0}, call_id="r1")],
            [function_call("escalate_to_human", {"handoff": draft}, call_id="e1")],
        ])
        agent = Agent(name="Desk", model=model, tools=[lookup_order, issue_refund, escalate_to_human],
                      tool_use_behavior=StopAtTools(stop_at_tool_names=["escalate_to_human"]))
        result = await Runner.run(agent, "refund all of it")
        out = finalize_handoff(dict(draft), "C-42", result.new_items, {"A-1001", "A-2002"})
        if not isinstance(out, HumanHandoff):
            record(4, False, "Return a HumanHandoff (use HumanHandoff.model_validate).")
            return
        want_actions = [
            'Called lookup_order({"order_id":"A-2002"})',
            '  -> {"status": "delivered", "total_usd": 640.0}',
            'Called issue_refund({"order_id":"A-2002","amount_usd":640.0})',
            '  -> {"error": "needs_human", "message": "Above the $500 limit"}',
        ]
        if out.customer_id != "C-42":
            record(4, False, "customer_id must come from the authenticated argument, not from the model.")
        elif "Issued a $640 refund" in out.actions_taken:
            record(4, False, "actions_taken still contains the model's claim; rebuild it from new_items.")
        elif any("handoff_received" in a or "escalate_to_human" in a for a in out.actions_taken):
            record(4, False, "Leave the escalate_to_human call and its output out of actions_taken.")
        elif out.actions_taken != want_actions:
            record(4, False, f"actions_taken should be {want_actions}, got {out.actions_taken}.")
        elif out.order_id is not None:
            record(4, False, "'A-2O02' (letter O) is not a known order: set order_id to None.")
        elif (out.issue_summary, out.amount_usd, out.escalation_reason) != (draft["issue_summary"], 640.0, "policy_limit"):
            record(4, False, "Keep the model-written fields (summary, amount, reason) unchanged.")
        else:
            draft2 = {**draft, "order_id": "A-1001"}
            out2 = finalize_handoff(draft2, "C-42", [], {"A-1001"})
            if out2.order_id != "A-1001" or out2.actions_taken != []:
                record(4, False, "A known order_id must be kept; no run items means no actions.")
            else:
                record(4, True)
    except NotImplementedError:
        record(4, False, "Implement finalize_handoff first.")
    except Exception as e:
        record(4, False, f"{type(e).__name__}: {e}")


await check_ex4()

### Exercise 5 — Per-field confidence thresholds

Write `pick_thresholds(rows, target_precision=0.98, min_support=50)`. `rows` is a labeled validation set: dicts with `field`, `confidence` and `correct` (bool). Return `{field: threshold_or_None}` where, **per field**, the threshold is the **lowest** distinct confidence value `t` such that the items with `confidence >= t`

- number at least `min_support`, and
- have precision (share correct) of at least `target_precision`.

Use `None` for a field where no such `t` exists. Precision is measured over **everything at or above** `t`, not per bin.

In [ ]:
def pick_thresholds(rows: list[dict], target_precision: float = 0.98, min_support: int = 50) -> dict[str, float | None]:
    # TODO
    raise NotImplementedError

In [ ]:
def _rows(field_name, score, n, n_correct):
    return [{"field": field_name, "confidence": score, "correct": i < n_correct} for i in range(n)]


def check_ex5() -> None:
    try:
        rows = (_rows("amount", 0.60, 50, 25) + _rows("amount", 0.80, 100, 97) + _rows("amount", 0.95, 200, 200)
                + _rows("iban", 0.90, 100, 80) + _rows("iban", 0.99, 30, 30)
                + _rows("date", 0.70, 60, 60) + _rows("date", 0.90, 60, 60))
        random.Random(3).shuffle(rows)
        got = pick_thresholds(rows, 0.98, 50)
        want = {"amount": 0.80, "iban": None, "date": 0.70}
        if not isinstance(got, dict) or set(got) != set(want):
            record(5, False, f"Return one entry per field: {sorted(want)}; got {got!r}.")
        elif got["iban"] is not None:
            record(5, False, "iban: only 30 items score 0.99, below min_support=50, so there is no safe threshold (None).")
        elif got["amount"] == 0.95:
            record(5, False, "amount: precision is measured over everything >= t. At t=0.80 that is 297/300 = 0.99, so 0.80 is the lowest safe threshold.")
        elif got["amount"] == 0.60:
            record(5, False, "amount: at t=0.60 precision is 322/350 = 0.92, below the target.")
        elif got != want:
            record(5, False, f"expected {want}, got {got}.")
        else:
            got2 = pick_thresholds(_rows("x", 0.5, 10, 10), 0.9, 5)
            mixed = _rows("y", 0.6, 50, 45) + _rows("y", 0.8, 50, 50)
            lo, hi = pick_thresholds(mixed, 0.90, 50), pick_thresholds(mixed, 0.98, 50)
            if got2 != {"x": 0.5}:
                record(5, False, f"Respect the min_support argument; got {got2}.")
            elif (lo, hi) != ({"y": 0.6}, {"y": 0.8}):
                record(5, False, "Use the target_precision argument: at t=0.6, 95/100 are correct, which meets a 0.90 "
                                 f"target but not 0.98 (then t=0.8). Expected {{'y': 0.6}} and {{'y': 0.8}}, got {lo} and {hi}.")
            else:
                record(5, True)
    except NotImplementedError:
        record(5, False, "Implement pick_thresholds first.")
    except Exception as e:
        record(5, False, f"{type(e).__name__}: {e}")


check_ex5()

### Exercise 6 — A stratified audit sampler

Write `stratified_sample(items, strata_keys, rate=0.02, min_per_stratum=20, seed=7)`. Group `items` (dicts) by the tuple of their `strata_keys` values. From each group take `min(len(group), max(min_per_stratum, round(rate * len(group))))` distinct items at random, using `random.Random(seed)`, so the same seed gives the same sample. Return one flat list.

In [ ]:
def stratified_sample(items: list[dict], strata_keys: tuple[str, ...], rate: float = 0.02,
                      min_per_stratum: int = 20, seed: int = 7) -> list[dict]:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> None:
    try:
        rng = random.Random(1)
        items = []
        for i in range(5000):
            doc = "invoice" if i < 3500 else "receipt" if i < 4850 else "handwritten_note"
            items.append({"id": i, "doc_type": doc, "field": rng.choice(["amount", "date"])})
        keys = ("doc_type", "field")
        sample = stratified_sample(items, keys, rate=0.02, min_per_stratum=20, seed=7)
        groups = Counter(tuple(i[k] for k in keys) for i in items)
        got = Counter(tuple(i[k] for k in keys) for i in sample)
        ids = [i["id"] for i in sample]
        if len(ids) != len(set(ids)):
            record(6, False, "The sample contains duplicates; sample without replacement.")
            return
        by_id = {i["id"]: i for i in items}
        if any(not isinstance(s, dict) or by_id.get(s.get("id")) is not s for s in sample):
            record(6, False, "Return the original item dicts from items (no copies, nothing else).")
            return
        for stratum, size in sorted(groups.items()):
            want = min(size, max(20, round(0.02 * size)))
            if got.get(stratum, 0) != want:
                record(6, False, f"stratum {stratum} (size {size}) should contribute {want} items, got {got.get(stratum, 0)}. "
                                 "Small strata need the minimum; large ones the proportional rate.")
                return
        for stratum, size in sorted(groups.items()):
            if size < 200:
                continue
            in_order = [i["id"] for i in items if tuple(i[k] for k in keys) == stratum]
            picked = {i["id"] for i in sample if tuple(i[k] for k in keys) == stratum}
            if picked == set(in_order[: len(picked)]):
                record(6, False, f"stratum {stratum}: you took its first {len(picked)} items. Errors often cluster by "
                                 "input order (time, batch, upload), so draw at random with random.Random(seed).")
                return
        again = stratified_sample(items, keys, rate=0.02, min_per_stratum=20, seed=7)
        if sorted(i["id"] for i in again) != sorted(ids):
            record(6, False, "The same seed must give the same sample (use random.Random(seed)).")
            return
        other = stratified_sample(items, keys, rate=0.02, min_per_stratum=20, seed=8)
        if sorted(i["id"] for i in other) == sorted(ids):
            record(6, False, "A different seed must give a different sample: use the seed argument in random.Random(seed).")
            return
        tiny = stratified_sample(items[:5], ("doc_type",), rate=0.5, min_per_stratum=20, seed=1)
        if len(tiny) != 5:
            record(6, False, "Never ask for more items than a stratum has.")
            return
        record(6, True)
    except NotImplementedError:
        record(6, False, "Implement stratified_sample first.")
    except Exception as e:
        record(6, False, f"{type(e).__name__}: {e}")


check_ex6()

## Part C — Architecture scenario

A car insurer wants a **claims agent** for minor damage claims:

- Customers submit claims with photos and a description in a web app. The agent can read the policy, the claim history and repair-shop quotes, and it can **approve payouts**.
- Regulations require that every payout above €2,000, and every claim with suspected fraud, is decided by a licensed adjuster. Adjusters work 8:00–18:00 and sometimes take two days.
- The agent runs on the Agents SDK in the insurer's cloud. A partner's damage-estimation service is exposed as a remote MCP server with read and write tools.
- The regulator requires an audit trail of every automated and human decision for seven years, and quarterly evidence that automated decisions are accurate per claim type.
- The current proposal is: "the prompt tells the agent to escalate if it is unsure or the customer is upset; adjusters get the chat transcript."

Describe your design: escalation triggers, where each rule is enforced, how approvals and multi-day pauses work, what the adjuster receives, how MCP is controlled, and how you prove accuracy to the regulator. Name at least two trade-offs.

*Your answer:*

-
-
-

<details><summary>Rubric: what a strong answer contains</summary>

- **Triggers replaced.** "Unsure" and "upset" are weak triggers. Observable ones instead: explicit request for a person, claim type or coverage not in policy (policy gap), payout above €2,000 (regulatory), fraud signals from deterministic rules or a measured classifier, repeated failures (`max_turns` + `error_handlers`), ambiguous policyholder identity.
- **Enforcement in code.** `approve_payout` gets a tool input guardrail (policy exists and is active, claim belongs to the authenticated customer, amount within coverage, no duplicate payout) and a check inside the payment service. Instructions explain the rules; they don't enforce them. Guardrails run on the first agent / on the function tool itself.
- **Approvals.** `needs_approval` callable: above €2,000 or any fraud flag → adjuster; fails closed on malformed arguments. `ToolExecutionConfig(pre_approval_tool_input_guardrails=True)` so adjusters only see payouts policy allows. Rejections carry a `rejection_message` the model can explain to the customer.
- **Multi-day pauses.** `RunState.to_string()` stored server-side with an opaque review ID, version marker and expiry; secrets kept out of the context (or a `context_serializer`), context rebuilt with `context_override`; adjusters authenticated and authorized server-side; decisions only for server-owned pending calls; atomic pending → resuming; expiry fails closed with a customer message. Note the 30-day retention if `previous_response_id` is used.
- **Adjuster handoff.** A strict handoff schema (claim, policy, amount, reason enum, evidence links, actions taken, recommendation), facts filled from the run and the claims DB, not from the model. Adjusters see source material (photos, quotes), not only the agent's claim.
- **MCP.** Partner server with `allowed_tools`; read tools `never`, write tools approved (`require_approval` split, `mcp_approval_request` / `mcp_approval_response`, or `HostedMCPTool` interruptions). No `on_approval_request` that auto-approves writes.
- **Accuracy evidence.** A labeled set per claim type; calibrated thresholds where a confidence gates automation; stratified sampling of auto-approved claims (claim type × amount band × channel) with per-stratum minimums; per-stratum accuracy in the quarterly report; corrections fed back into the eval set.
- **Audit trail.** Every approval request and decision logged (who, when, arguments, decision, message, tool result) plus traces; retention of seven years in your own store; separation of duties (an adjuster can't approve their own or a relative's claim).
- **Trade-offs named**, for example: more approvals (safety, regulation) vs customer wait time and adjuster load; regex fraud rules (cheap, explainable) vs a classifier (recall, needs labeled data); hosted MCP approvals vs running the partner client yourself for more control.
</details>

## Scorecard

In [ ]:
quiz_correct, quiz_total = QUIZ_SCORE["correct"], QUIZ_SCORE["total"]
ex_total = 6
ex_passed = sum(EX_RESULTS.get(n, False) for n in range(1, ex_total + 1))
overall = (quiz_correct + ex_passed) / (quiz_total + ex_total)
print(f"Quiz:      {quiz_correct}/{quiz_total}")
print(f"Exercises: {ex_passed}/{ex_total}  " + " ".join(f"Ex{n}:{'✓' if EX_RESULTS.get(n) else '✗'}" for n in range(1, ex_total + 1)))
print(f"Overall:   {overall:.0%}  ->  {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%)")
print("Part C is self-assessed against the rubric.")

Next chapter: [Chapter 10 — Error handling in multi-agent systems](../10-multi-agent-error-handling/README.md)

---

## Solutions (spoilers)

<details><summary>Quiz answers with explanations</summary>

- **Q1 — C.** Ambiguous identity: ask for another identifier and never guess. A and D guess (and could change the wrong customer's address); B escalates something the agent can resolve with one question.
- **Q2 — B.** `StopAtTools` ends the run deterministically after the turn in which the escalate tool fires: the model gets no further call, and `final_output` is the tool's return value. It does not cancel other tool calls in that **same** response, so also set `ModelSettings(parallel_tool_calls=False)` and block side effects in code when a response escalates (practice notebook, 9.3). A is a probabilistic instruction; C breaks every normal multi-step flow; D forces more tool calls, not fewer.
- **Q3 — D.** `error_handlers={"max_turns": ...}` turns the cap into a controlled escalation with the run's items for the handoff. A and B only move or remove the safety cap; C repeats the same failing loop and its side effects.
- **Q4 — A.** Input guardrails run only on the first agent's input. Run-level guardrails in `RunConfig` also apply to the first agent. B changes timing, not whether it runs; C and D are false.
- **Q5 — C.** `rejection_message` is the exact text the model reads in place of the tool result, in the same paused run. A tampers with the snapshot; B starts a new turn and loses the pending call; D executes a call the supervisor rejected.
- **Q6 — D.** Sticky decisions apply to later calls of the same tool by the same agent for the rest of the run and are stored in the run state, so they survive `to_string()` / `from_string()`. They don't reach other runs (A), other tools (B), and they are more than one call (C).
- **Q7 — B.** The context is serialized with the snapshot, so a secret in it lands in the database. `context_override` (A) replaces the context on load but does not remove it from an already-written blob; base64 (C) is encoding; `strict_context` (D) only enforces a mapping or a serializer.
- **Q8 — A.** Each snapshot can be resumed any number of times; only your store can guarantee once. An idempotency key on the voucher service is a good second layer. B, C and D don't address replays.
- **Q9 — D.** Calibration is per field, on labeled data, with a minimum support; a field that never meets the target goes to review. A ignores what the data says; B is a prompt hope; C throws away the fields that are reliable.
- **Q10 — B.** Approvals are answered with an `mcp_approval_response` input item in a follow-up that continues the conversation; the API then calls the server. `authorization` is never stored, so the tool config must be sent again. A and D bypass the approval loop; C removes the control.
</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
COVERED_TOPICS = {"late_delivery", "damaged", "refund", "address_change", "duplicate_charge", "order_status"}
HUMAN_REQUEST = re.compile(
    r"\b(want|need|talk to|speak to|speak with|get me|connect me|transfer me)\b"
    r"[^.?!]*?\b(human|person|manager|supervisor)\b", re.I)


def route(case: dict) -> str:
    if HUMAN_REQUEST.search(case["text"]):
        return "escalate:customer_requested_human"
    if case["matches"] > 1:
        return "ask_identifier"
    if case["topic"] not in COVERED_TOPICS:
        return "escalate:policy_gap"
    if case["clarifications"] >= 2:
        return "escalate:cannot_progress"
    if case["refund_usd"] > 500:
        return "escalate:policy_limit"
    if case["refund_usd"] > 100:
        return "approval"
    return "resolve"
```
</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
async def refund_needs_approval(ctx, params: dict, call_id: str) -> bool:
    amount = params.get("amount_usd")
    if isinstance(amount, bool) or not isinstance(amount, (int, float)):
        return True                                   # missing or not a number: fail closed
    if not math.isfinite(amount) or amount < 0:
        return True                                   # NaN, infinity, negative: fail closed
    if ctx.context.account_age_days < 30:
        return True
    return amount > ctx.context.auto_limit_usd
```
</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
def apply_review(store: dict, review_id: str, reviewer: dict, decisions: dict, now: datetime) -> dict[str, str]:
    record = store[review_id]                                         # KeyError if unknown
    if reviewer.get("role") != "supervisor" or reviewer.get("user_id") == record["customer_id"]:
        raise PermissionError("reviewer may not decide this review")
    if record["status"] != "pending":
        raise RuntimeError(f"review already {record['status']}")
    unknown = set(decisions) - set(record["pending_call_ids"])
    if unknown:
        raise ValueError(f"not pending: {sorted(unknown)}")
    if now > record["expires_at"]:
        record["status"] = "expired"
        return {cid: "reject" for cid in record["pending_call_ids"]}
    record["status"] = "resuming"                                     # one atomic update in a real database
    return {cid: "approve" if decisions.get(cid) is True else "reject" for cid in record["pending_call_ids"]}
```
</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
def finalize_handoff(draft: dict, customer_id: str, new_items, known_orders: set[str]) -> HumanHandoff:
    actions, escalation_calls = [], set()
    for item in new_items:
        if item.type == "tool_call_item":
            if getattr(item.raw_item, "type", None) != "function_call":
                continue                                  # hosted/built-in call: no name or arguments to report
            if item.raw_item.name == "escalate_to_human":
                escalation_calls.add(item.raw_item.call_id)
            else:
                actions.append(f"Called {item.raw_item.name}({item.raw_item.arguments})")
        elif item.type == "tool_call_output_item" and item.raw_item.get("call_id") not in escalation_calls:
            actions.append(f"  -> {str(item.output)[:200]}")
    data = dict(draft)
    data["customer_id"] = customer_id
    data["actions_taken"] = actions
    if data.get("order_id") not in known_orders:
        data["order_id"] = None
    return HumanHandoff.model_validate(data)
```
</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
def pick_thresholds(rows: list[dict], target_precision: float = 0.98, min_support: int = 50) -> dict[str, float | None]:
    by_field: dict[str, list[tuple[float, bool]]] = defaultdict(list)
    for r in rows:
        by_field[r["field"]].append((r["confidence"], r["correct"]))
    out: dict[str, float | None] = {}
    for fld, scored in by_field.items():
        out[fld] = None
        for t in sorted({s for s, _ in scored}):
            above = [ok for s, ok in scored if s >= t]
            if len(above) < min_support:
                break
            if sum(above) / len(above) >= target_precision:
                out[fld] = t
                break
    return out
```
</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
def stratified_sample(items: list[dict], strata_keys: tuple[str, ...], rate: float = 0.02,
                      min_per_stratum: int = 20, seed: int = 7) -> list[dict]:
    rng = random.Random(seed)
    groups: dict[tuple, list[dict]] = defaultdict(list)
    for item in items:
        groups[tuple(item[k] for k in strata_keys)].append(item)
    sample = []
    for _, group in sorted(groups.items()):
        n = min(len(group), max(min_per_stratum, round(rate * len(group))))
        sample.extend(rng.sample(group, n))
    return sample
```
</details>